# Mean-Reversion Pairs Trading Strategy
Limit order book tick data (10-second intervals) spanning Nov 2019 – Dec 2020.
We construct a spread between two instruments, confirm stationarity via ADF test,
and evaluate a z-score-based mean-reversion strategy out-of-sample.

In [7]:
import pandas as pd

df = pd.read_csv("final_data_10s.csv")
df["Time"] = pd.to_datetime(df["Time"])
df = df.set_index("Time")

We estimate the instrument's true price with the midpoint of the buyer's bid and ask prices - ask prices are slightly inflated, whereas bid prices are deflated.

In [8]:
df["X_mid"] = (df["X_ASK"] + df["X_BID"]) / 2
df["Y_mid"] = (df["Y_ASK"] + df["Y_BID"]) /2

### Cointegration test
We use the Engle-Granger cointegration test to determine whether a linear combination of X and Y is stationary. If they are, the two instruments will, on average, move together in the long run and any divergence is temporary.

In [9]:
from statsmodels.tsa.stattools import coint

X = df['X_mid'].resample("1D").last().dropna()
Y = df['Y_mid'].resample("1D").last().dropna()
score, p_value, _ = coint(X, Y)
print(f"p_value: {p_value}")

p_value: 0.07938604033139546


1 minute: p_value = 0.10141799457880102

1 Day: p_value = 0.07938604033139546

Neither frequency clears the 0.05 threshold, suggesting X and Y are not formally cointegrated. We resample to daily bars as high-frequency microstructure noise reduces the reliability of the test. 

### Spread Construction
The Engle-Granger test fell short of the 0.05 threshold at all frequencies, 
suggesting these instruments are not formally cointegrated. However, we can still 
test whether the spread between them is stationary, which would be a direct and sufficient 
condition for a mean-reversion strategy, without requiring strict cointegration.

We decompose the Engle-Granger procedure manually: first fitting an OLS regression 
to extract the hedge ratio, then running ADF directly on the residuals. This gives 
us both the spread series needed for trading and a stationarity test on it.

The hedge ratio scales X so that the spread captures only the deviation in the 
pricing relationship between X and Y, not their shared price-level differences.

In [10]:
from sklearn.linear_model import LinearRegression

train_df = df[:"2020-06-30"]

model = LinearRegression()
model.fit(train_df[["X_mid"]], train_df["Y_mid"])
hedge_ratio = model.coef_[0]

train_df["spread"] = train_df["Y_mid"] - hedge_ratio * train_df["X_mid"] - model.intercept_

/var/folders/_s/1xjrcgxx5lx_x_jy1mjp6ly80000gn/T/ipykernel_42843/3353763049.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["spread"] = train_df["Y_mid"] - hedge_ratio * train_df["X_mid"] - model.intercept_


In [11]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(train_df["spread"].resample("1D").last().dropna())
print(f"ADF p-value: {result[1]}")

ADF p-value: 0.0495720142423908


Unlike `coint()`, the standard ADF test does not apply corrected critical values that account for the fact that residuals come from a fitted regression. The result (p=0.0094) therefore cannot be used to claim formal cointegration — but it does provide direct evidence that the spread itself is stationary, which is a sufficient condition for a mean-reversion strategy.

### Mean Reversion Strategy

ADF p-value: 0.009393366693707987 < 0.05

Evidence that the spread is stationary and mean-reverting

In [12]:
window = 60

spread_daily = df["spread"].resample("1D").last().dropna()

train = spread_daily[:"2020-06-30"]
test = spread_daily["2020-07-01":]

frozen_mean = train.rolling(window, min_periods=window).mean().iloc[-1]
frozen_std = train.rolling(window, min_periods=window).std().iloc[-1]

z_score = (test - frozen_mean) / frozen_std

long_entry = z_score < -2
short_entry = z_score > 2
exit = z_score.abs() < 0.5

KeyError: 'spread'

### Signal Generation (Out-of-Sample)
The dataset is split into a training period (Nov 2019 – Jun 2020) and a held-out 
test period (Jul – Dec 2020). Rolling mean and standard deviation are estimated on 
the training period only, and the final values are fixed as parameters applied to 
the test period to make sure that the strategy never sees future data.

The spread is normalised into a z-score measuring how many standard deviations the 
current spread sits from its recent mean. We enter long when z < -2 (spread 
abnormally low, expected to revert upward), short when z > +2 (spread abnormally 
high, expected to revert downward), and exit when the spread returns within 0.5 
standard deviations of the mean.

In [ ]:
import numpy as np

positions = pd.Series(np.nan, index=test.index)
positions[long_entry] = 1
positions[short_entry] = -1
positions[exit] = 0
positions = positions.ffill().fillna(0)

spread_pnl = spread_daily.diff()
X_daily = df["X_mid"].resample("1D").last().dropna()
Y_daily = df["Y_mid"].resample("1D").last().dropna()
notional = Y_daily + abs(hedge_ratio) * X_daily
spread_returns = spread_pnl / notional.shift(1)
strategy_returns = positions.shift(1) * spread_returns

cumulative = (1 + strategy_returns).cumprod()
cumulative.plot(title="Mean-Reversion Strategy - Cumulative Returns")

### Backtest
Entry signals are forward-filled to hold positions until an exit signal fires. 
`positions.shift(1)` ensures returns are calculated on the day *after* observing 
the signal, preventing lookahead bias. Strategy returns are the product of the 
lagged position and the daily spread return.

In [ ]:
strat_returns_oos = strategy_returns.loc[test.index]

sharpe = strat_returns_oos.mean() / strat_returns_oos.std() * np.sqrt(252)
max_dd = (cumulative / cumulative.cummax() - 1).min()

in_market_returns = strat_returns_oos[positions.shift(1) != 0]
hit_rate = (in_market_returns > 0).mean()

print(f"Sharpe:       {sharpe:.2f}")
print(f"Max Drawdown: {max_dd:.2%}")
print(f"Hit Rate:     {hit_rate:.2%}")

### Transaction Costs
A transaction cost of 1 basis point (0.01%) is applied on each position change. 
`positions.diff().abs()` identifies entry and exit days and deducts the cost from 
returns on those days only.

In [ ]:
transaction_cost_bps = 0.0001
trades = positions.diff().abs().fillna(0)
cost_impact = (trades > 0).astype(float) * transaction_cost_bps * 2

strategy_returns_net = strategy_returns.reindex(test.index) - cost_impact
sharpe_net = strategy_returns_net.mean() / strategy_returns_net.std() * np.sqrt(252)
print(f"Sharpe (after costs): {sharpe_net:.2f}")